# ACE-Step-Studio (Colab T4 Full Runner)\n\nΑυτό το notebook είναι για **Google Colab T4 (16GB)** και προσπαθεί να τρέξει το project end-to-end.\n\n## Τι κάνει\n1. Ελέγχει GPU\n2. Κάνει clone/update το repo\n3. Εγκαθιστά dependencies (best-effort)\n4. Κάνει auto-detect entrypoint\n5. Τρέχει το project\n6. Μαζεύει outputs και τα κάνει zip για download\n

In [ ]:
#@title 1) GPU Check (T4 expected)\n!nvidia-smi\nimport torch\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))\n

In [ ]:
#@title 2) Clone / Update repository\nimport os, subprocess, sys\nREPO_URL = 'https://github.com/tzomaik-art/ACE-Step-Studio-Colab-Optimized.git'\nREPO_DIR = '/content/ACE-Step-Studio-Colab-Optimized'\n\nif not os.path.exists(REPO_DIR):\n    !git clone {REPO_URL} {REPO_DIR}\nelse:\n    %cd {REPO_DIR}\n    !git pull\n\n%cd {REPO_DIR}\nprint('Repo ready at', REPO_DIR)\n

In [ ]:
#@title 3) Install deps (best-effort for T4)\nimport os, sys, subprocess, textwrap\n\ndef run(cmd):\n    print('\n>>>', cmd)\n    r = subprocess.run(cmd, shell=True)\n    if r.returncode != 0:\n        print(f'[warn] command failed: {cmd}')\n    return r.returncode\n\n# Base tools\nrun('pip -q install --upgrade pip setuptools wheel')\nrun('pip -q install ninja packaging')\n\n# Project-specific requirements\nif os.path.exists('requirements.txt'):\n    run('pip -q install -r requirements.txt')\nelif os.path.exists('pyproject.toml'):\n    run('pip -q install .')\nelse:\n    print('[info] No requirements.txt/pyproject.toml found. Installing common ML stack...')\n    run('pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')\n    run('pip -q install diffusers transformers accelerate safetensors xformers gradio')\n\nprint('Dependency step finished.')\n

In [ ]:
#@title 4) Optional: Mount Google Drive\nUSE_DRIVE = False  #@param {type:'boolean'}\nif USE_DRIVE:\n    from google.colab import drive\n    drive.mount('/content/drive')\n    print('Drive mounted at /content/drive')\nelse:\n    print('Drive mount skipped')\n

In [ ]:
#@title 5) Auto-detect entrypoint\nimport os, glob\n\ncandidates = [\n    'app.py', 'main.py', 'inference.py', 'run.py', 'launch.py',\n    'webui.py', 'server.py'\n]\n\nfound = None\nfor c in candidates:\n    if os.path.exists(c):\n        found = c\n        break\n\nif found is None:\n    # fallback: first python file in root\n    py_files = [p for p in glob.glob('*.py') if not p.startswith('_')]\n    found = py_files[0] if py_files else None\n\nprint('Detected entrypoint:', found)\nENTRYPOINT = found\n

In [ ]:
#@title 6) Run project\nimport os, subprocess, shlex\n\nEXTRA_ARGS = ''  #@param {type:'string'}\n\nif ENTRYPOINT is None:\n    raise RuntimeError('No entrypoint found. Please set manually.')\n\ncmd = f'python {shlex.quote(ENTRYPOINT)} {EXTRA_ARGS}'\nprint('Running:', cmd)\nsubprocess.run(cmd, shell=True, check=False)\n

In [ ]:
#@title 7) Collect outputs -> zip -> download\nimport os, zipfile, glob\nfrom google.colab import files\n\npossible_output_dirs = ['outputs', 'output', 'results', 'runs', 'artifacts']\nto_zip = []\n\nfor d in possible_output_dirs:\n    if os.path.isdir(d):\n        to_zip.append(d)\n\n# Also include common generated media in root\nroot_media = []\nfor ext in ['*.png', '*.jpg', '*.jpeg', '*.mp4', '*.wav', '*.mp3', '*.gif']:\n    root_media.extend(glob.glob(ext))\n\nzip_name = '/content/ace_step_studio_outputs.zip'\nwith zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:\n    for d in to_zip:\n        for root, _, files_ in os.walk(d):\n            for f in files_:\n                p = os.path.join(root, f)\n                zf.write(p, arcname=p)\n    for f in root_media:\n        if os.path.isfile(f):\n            zf.write(f, arcname=f)\n\nprint('Created:', zip_name)\nfiles.download(zip_name)\n